In [ ]:
import matplotlib.pyplot as plt
from google.cloud import bigquery

client = bigquery.Client(project="prc-data-challenge-2025")


def q(sql):
    return client.query(sql).to_dataframe()

In [ ]:
# fuel_train flights missing from flightlist_train
q("""
SELECT
  COUNT(DISTINCT ft.flight_id) AS n_fuel_flights,
  COUNT(DISTINCT IF(flt.flight_id IS NULL, ft.flight_id, NULL)) AS n_flights_without_flightlist
FROM `prc-data-challenge-2025.prc2025.fuel_train` AS ft
LEFT JOIN `prc-data-challenge-2025.prc2025.flightlist_train` AS flt
  ON ft.flight_id = flt.flight_id
""")

LEFT JOIN fuel_train -> flightlist_train (distinct flight_id): 11037 flights in fuel_train, 0 missing from flightlist_train.
Same count on both sides (11037) -> fuel_train and flightlist_train have exactly the same flight_id.

In [ ]:
# flights without trajectory
q(
    """
    SELECT COUNT(DISTINCT flt.flight_id) as flight_count, COUNT(DISTINCT IF(ft.flight_id IS NULL, flt.flight_id, NULL)) as n_flights_without_trajectory
    FROM `prc-data-challenge-2025.prc2025.flightlist_train` as flt
    LEFT JOIN(
    SELECT DISTINCT flight_id,
    FROM `prc-data-challenge-2025.prc2025.flights_train`
    ) as ft
    ON ft.flight_id = flt.flight_id
"""
)

The 3 tables (fuel_train, flightlist_train, flights_train) share exactly the same 11037 flight_id.
-> joins on flight_id lose nothing.

In [ ]:
# fuel interval coverage vs takeoff/landing
df = q("""
WITH fuel_per_flight AS (
  SELECT
    flight_id,
    COUNT(*) AS n_intervals,
    TIMESTAMP_MICROS(DIV(MIN(start), 1000)) AS first_start,
    TIMESTAMP_MICROS(DIV(MAX(`end`), 1000)) AS last_end
  FROM `prc-data-challenge-2025.prc2025.fuel_train`
  GROUP BY flight_id
)
SELECT
  fl.flight_id,
  f.n_intervals,
  TIMESTAMP_DIFF(f.first_start, fl.takeoff, MINUTE) AS start_after_takeoff_min,
  TIMESTAMP_DIFF(fl.landed, f.last_end, MINUTE) AS end_before_landing_min
FROM fuel_per_flight AS f
JOIN `prc-data-challenge-2025.prc2025.flightlist_train` AS fl
  ON f.flight_id = fl.flight_id
""")

df.describe()

Fuel intervals vs takeoff/landed (one row per flight, minutes):
- n_intervals: 1 to 98, median 10.
- first interval starts after takeoff: median 81 min, Q1 6 min, max 846 min.
- last interval ends before landing: median 52 min, Q1 5 min, max 770 min.
-> intervals are chosen windows of the flight, not a full split: we predict fuel per window, not per flight.
- negative values: min -242 (starts 4 h before takeoff) and -169 (ends ~3 h after landing).
  A few minutes to tens of minutes = plausible (taxi, APU on ground); hours = suspicious (wrong takeoff/landed or interval times?).
-> to count: how many flights have negative values, and below -30 min.

In [ ]:
# intervals outside the flight (before takeoff / after landing)
print("end_before_landing_min <0:", (df["end_before_landing_min"] < 0).sum())
print("start_after_takeoff_min <0:", (df["start_after_takeoff_min"] < 0).sum())
print("end_before_landing_min < -30:", (df["end_before_landing_min"] < -30).sum())
print("start_after_takeoff_min < -30:", (df["start_after_takeoff_min"] < -30).sum())

Intervals outside the flight, by size:
- before takeoff: 698 flights, of which only 96 (0.9%) by more than 30 min.
- after landing: 825 flights, of which only 86 (0.8%) by more than 30 min.
-> most overlaps are short = ground fuel burn (taxi, APU), normal.
-> ~1% of flights with > 30 min overlap: suspicious (wrong times?), to keep in mind.
-> ground parts of intervals have no trajectory (trajectory starts in the air): ground burn must be estimated another way.

In [ ]:
# used airports missing from airports or without elevation
q(
    """
WITH used_airports AS 
(
    SELECT origin_icao as icao, 
    FROM `prc-data-challenge-2025.prc2025.flightlist_train`
    UNION DISTINCT
    SELECT destination_icao as icao,
    FROM `prc-data-challenge-2025.prc2025.flightlist_train`
)
SELECT
  COUNT(*) AS n_used_airports,
  COUNTIF(a.icao IS NULL) AS n_missing_from_airports,
  COUNTIF(a.icao IS NOT NULL AND a.elevation IS NULL) AS n_null_elevation
FROM used_airports as u 
LEFT JOIN `prc-data-challenge-2025.prc2025.airports` AS a
ON u.icao = a.icao
"""
)

Airports used in flightlist_train (origin or destination, UNION DISTINCT): 320
(273 origins + 260 destinations, 213 used both ways).
0 missing from airports, 0 with null elevation.
-> the 219 airports without elevation are never used(train part): no issue for our flights.

In [ ]:
# average fuel rate per flight with aircraft type
df = q(
    """
WITH fuel_data_per_flight as (
    SELECT flight_id, COUNT(*) AS n_intervals,
    SUM(fuel_kg) / SUM((`end` - start) / 1e9) AS avg_fuel_rate_kg_s
    FROM `prc-data-challenge-2025.prc2025.fuel_train`
    GROUP BY flight_id
)
SELECT fdpf.flight_id,flt.aircraft_type,fdpf.n_intervals,fdpf.avg_fuel_rate_kg_s
FROM fuel_data_per_flight as fdpf
JOIN `prc-data-challenge-2025.prc2025.flightlist_train` as flt
ON fdpf.flight_id = flt.flight_id
"""
)

In [ ]:
# fuel rate per flight, by aircraft type (boxplot)
fig, ax = plt.subplots(figsize=(14, 6))
df.boxplot(column="avg_fuel_rate_kg_s", by="aircraft_type", ax=ax, rot=90)
fig.suptitle("")
ax.set_title("Average fuel rate per flight, by aircraft type")
ax.set_xlabel("aircraft type")
ax.set_ylabel("average fuel rate (kg/s)")
plt.show()

Outlier definition: "far out" points (Tukey): beyond Q3 + 3*IQR or below Q1 - 3*IQR, per aircraft type.
Visually: B744 (~13 kg/s, maybe the 5-6.5 group) and A333 (~11 kg/s). A332 / B77W points near the whisker are not kept.
Types with very few flights (A318, B39M, B77L, MD11, A388...) cannot be judged.

In [ ]:
# Tukey far-out bounds per aircraft type and outlier flights
df["q1"] = df.groupby("aircraft_type")["avg_fuel_rate_kg_s"].transform(
    lambda s: s.quantile(0.25)
)
df["median"] = df.groupby("aircraft_type")["avg_fuel_rate_kg_s"].transform(
    lambda s: s.quantile(0.50)
)
df["q3"] = df.groupby("aircraft_type")["avg_fuel_rate_kg_s"].transform(
    lambda s: s.quantile(0.75)
)
df["iqr"] = df["q3"] - df["q1"]
df["lower_bound"] = df["q1"] - 3 * df["iqr"]
df["upper_bound"] = df["q3"] + 3 * df["iqr"]

df.head()

outliers = df[
    (df["avg_fuel_rate_kg_s"] < df["lower_bound"])
    | (df["avg_fuel_rate_kg_s"] > df["upper_bound"])
]
outliers.sort_values("avg_fuel_rate_kg_s", ascending=False)[
    [
        "flight_id",
        "aircraft_type",
        "avg_fuel_rate_kg_s",
        "median",
        "lower_bound",
        "upper_bound",
    ]
]

Far-out flights (Tukey 3*IQR per aircraft type): 205 flights (1.9%).
High side: prc790175280 and prc784170877 (B744, ~13 kg/s, ~5x median), prc783282112 (A333, 11.1 kg/s, ~8x median),
  new one: prc788278172 (A332, 6.6 kg/s, ~4x median).
Low side too: prc780561659 (A359, 0.14 kg/s, ~12x below median), several A359 / B77W at ~0.3 kg/s.
Limits: lower bound can be negative (A333) -> no low outlier possible; 1-flight types (iqr = 0) cannot be judged.

In [ ]:
# outlier flights per aircraft type
outliers["aircraft_type"].value_counts()

Far-out flights per type, as share of the type's flights:
B744 14% (13/90), B788 11% (65/583), B77W 7%, B789 5%, A332 4.5%, A359 3%, A321 2%, A20N 0.3%, A320 0.2%.
-> A320 family almost clean; widebodies much more affected.
-> B788 11%: too many for isolated errors. Narrow distribution (small IQR) -> rule too strict, or a real subgroup (airline / source)?
-> Tukey flags candidates, not proven errors: only the extreme cases (B744 / A333 ~5-12x median) are clearly wrong.

In [ ]:
# outliers too high / too low
print("too high:", (outliers["avg_fuel_rate_kg_s"] > outliers["upper_bound"]).sum())
print("too low:", (outliers["avg_fuel_rate_kg_s"] < outliers["lower_bound"]).sum())

Far-out flights: 115 too high, 90 too low (total 205, check ok).
-> both sides affected: not only inflated values, also under-estimated fuel -> general measurement quality issue (units / gauges / sources).

In [ ]:
# gap_ratio per route (top 20 routes)
df_routes = q(
    """
WITH gaps AS (
  SELECT
    flight_id,
    `timestamp`,
    (`timestamp` - LAG(`timestamp`) OVER (PARTITION BY flight_id ORDER BY `timestamp`)) 
    / 1e9 AS gap_s
  FROM `prc-data-challenge-2025.prc2025.flights_train`
), per_flight as(
SELECT flight_id,
  COUNT(*) AS n_points,
  (MAX(`timestamp`) - MIN(`timestamp`)) / 1e9 AS span_s,
  MAX(gap_s) AS max_gap_s,
  SUM(IF(gap_s > 600, gap_s, 0)) AS total_gap_s,
  SUM(IF(gap_s > 600, gap_s, 0)) / ((MAX(`timestamp`) - MIN(`timestamp`)) / 1e9) AS gap_ratio
FROM gaps 
GROUP BY flight_id
)
SELECT
  fl.origin_icao,
  fl.destination_icao,
  COUNT(*) AS n_flights,
  APPROX_QUANTILES(p.span_s / 3600, 4)[OFFSET(2)] AS median_span_h,
  APPROX_QUANTILES(p.gap_ratio, 4)[OFFSET(1)] AS q1_gap_ratio,
  APPROX_QUANTILES(p.gap_ratio, 4)[OFFSET(2)] AS median_gap_ratio,
  APPROX_QUANTILES(p.gap_ratio, 4)[OFFSET(3)] AS q3_gap_ratio
FROM per_flight AS p
JOIN `prc-data-challenge-2025.prc2025.flightlist_train` AS fl
  ON p.flight_id = fl.flight_id
GROUP BY fl.origin_icao, fl.destination_icao
ORDER BY n_flights DESC
LIMIT 20
"""
)

In [ ]:
# show gap_ratio per route
df_routes

gap_ratio per route (top 20 routes by n_flights):
- within a route, gap_ratio is very stable (e.g. SKBO-LEMD Q1-Q3 0.75-0.80, MSLP-KLAX 0.47-0.54) -> clusters = recurring routes. Confirmed.
- clusters found: ~4 h / 0.3 = MSLP <-> KIAD; ~9 h / 0.8 = SKBO <-> LEMD, ZSPD -> NZAA; ~4.7 h / 0.5 = MSLP <-> KLAX.
- North America internal routes (MMGL-KLAX, KBOS-KMCO...): 0 gaps -> good ADS-B coverage; gaps come from oceans / some regions.
- outbound and return routes have the same ratio: the path matters, not the direction.
- most frequent routes are Central/South America, US, China-Oceania: not a Europe-centred dataset.
-> gaps are predictable by route: could anticipate test-set gaps and fill gaps using other flights of the same route.

In [ ]:
# trajectory points inside each fuel interval
df = q("""
SELECT
  ft.idx,
  COUNT(flt.`timestamp`) AS n_points,
  (ft.`end` - ft.start) / 1e9 AS duration_s
FROM `prc-data-challenge-2025.prc2025.fuel_train` AS ft
LEFT JOIN `prc-data-challenge-2025.prc2025.flights_train` AS flt
  ON ft.flight_id = flt.flight_id
  AND flt.`timestamp` BETWEEN ft.start AND ft.`end`
GROUP BY ft.idx, ft.start, ft.`end`
""")
len(df)

In [ ]:
# share of fuel intervals with no trajectory point
n_zero = (df["n_points"] == 0).sum()
print("intervals with 0 points:", n_zero, "/", len(df), "=", n_zero / len(df))

df["n_points"].describe()

Intervals with 0 trajectory points: 3051 / 131530 = 2.3%.
But Q1 = 2 -> > 20% of intervals have only 1-2 points.
-> "0 points" underestimates the issue: an interval with 1-2 points has no usable trajectory either.
-> need a better coverage measure (points relative to interval duration?).

n_points per fuel interval: Q1 = 2, median 289, Q3 592, max 7011.
-> at least 25% of intervals have <= 2 trajectory points: practically no trajectory.
-> median interval (~5 min) has ~1 point/s: well covered.
-> two populations: almost empty vs well covered intervals.

In [ ]:
# trajectory density per interval
df["density"] = df.n_points / df.duration_s
df.density.describe()

density = n_points / duration_s (points per second) per fuel interval:
Q1 0.0067 (~1 point every 2.5 min), median 1.19, Q3 1.94, max 4.05.
-> at least 25% of intervals have practically no trajectory; half are well tracked (> 1 point/s).
-> big jump between Q1 and median: two populations (empty vs covered), few in between.

In [ ]:
# distribution of trajectory density per fuel interval
fig, ax = plt.subplots(figsize=(14, 6))
df.hist(column="density", bins=150, ax=ax)
fig.suptitle("")
ax.set_title("Trajectory density per fuel interval")
ax.set_xlabel("density (trajectory points per second)")
ax.set_ylabel("number of intervals")
plt.show()

Histogram of trajectory density per fuel interval (points/s):
- huge bar at ~0: ~48000 intervals (~37%) with practically no trajectory -> answer to Q6 (much more than the 2.3% with exactly 0 points).
- peak at ~2 points/s: ADS-B position messages are sent ~2 times per second -> density ~2 = fully covered interval (natural max, nothing above ~2.1).
- low flat plateau in between: partially covered intervals.
-> density / 2 ~ share of the interval covered by the trajectory (no magic threshold needed).

In [ ]:
# share of intervals below density thresholds
print("% of intervals with density < 0.5:", (df["density"] < 0.5).sum() / len(df) * 100)
print(
    "% of intervals with density < 0.08:", (df["density"] < 0.08).sum() / len(df) * 100
)

Coverage threshold from the histogram: valley at ~0.5 points/s (~25% of the interval covered, peak = 2 points/s).
density < 0.5: 54808 intervals (41.7%); density < 0.08 (first bar): 48194 (36.6%) -> threshold not critical (5 points apart).
-> Q6 answer: ~4 intervals out of 10 have (almost) no trajectory.
-> model cannot rely only on trajectory features: needs always-available features (interval duration, aircraft type, route,
   time since takeoff / before landing to infer flight phase), and possibly gap filling with other flights of the same route (Q5).